In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.gold;

In [0]:
from pyspark.sql.functions import col, sum, count, when, row_number
from pyspark.sql.window import Window

In [0]:
race_results_df = spark.read.table("f1.gold.race_results") 

In [0]:
constructor_standings_df = race_results_df.groupBy(
    "year",
    "constructor_id",
    "constructor_name",
    "constructor_ref",
    "constructor_nationality"
).agg(
    sum("points").alias("total_points"),
    count(when(col("position") == 1, True)).alias("wins"),
    count(when(col("position") <= 3, True)).alias("podiums"),
    count(when(col("position").isNotNull(), True)).alias("races_finished")
)

In [0]:
constructor_window_df = Window.partitionBy("year").orderBy(col("total_points").desc(), col("wins").desc())
constructor_standings_final_df = constructor_standings_df.withColumn("championship_position", row_number().over(constructor_window_df))

In [0]:
constructor_standings_final_df = constructor_standings_final_df.select(

    "year",

    "championship_position",

    "constructor_id",
    "constructor_name",
    "constructor_ref",
    "constructor_nationality",

    "total_points",
    "wins",
    "podiums",
    "races_finished"
)

In [0]:
constructor_standings_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year") \
    .saveAsTable("f1.gold.constructor_standings")

In [0]:
%sql

SELECT *
FROM f1.gold.constructor_standings
ORDER BY year DESC, championship_position ASC
LIMIT 20;